# NYC Yellow Taxi — Trip Duration Prediction (Tracked)

Same pipeline as `01-pipeline-untracked.ipynb`. Same data, same models, same code.

**What's different:** every model training run is now logged to MLflow. Every metric is queryable. Every model is versioned. The best model is registered, aliased, and loadable by name — not by file path.

Open a terminal and run this **before** executing the notebook:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then open http://localhost:5000 in your browser. Leave it open as you run this notebook.

## 1. Setup + MLflow Backend

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import joblib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from mlflow.models import infer_signature

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

print(f"mlflow {mlflow.__version__}")

mlflow 3.2.0


In [2]:
# Point MLflow at a SQLite database in this directory
MLFLOW_DB = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

# All runs in this notebook go under one experiment
EXPERIMENT_NAME = "nyc_taxi_duration"
mlflow.set_experiment(EXPERIMENT_NAME)

# Client for registry operations later
client = MlflowClient()

print(f"Tracking URI: sqlite:///{MLFLOW_DB}")
print(f"Experiment:   {EXPERIMENT_NAME}")

2026/10/04 22:22:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/04 22:22:14 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

Tracking URI: sqlite:////home/silva/SILVA.AI/Projects/MLOps/DDODS/Module_2_Reproducibility/Lesson_2_Experiment_Tracking_MLflow/mlflow.db
Experiment:   nyc_taxi_duration


In [3]:
class Config:
    """Same as the untracked notebook."""

    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    VAL_SIZE = 0.2
    CV_FOLDS = 5
    N_JOBS = -1

    MODEL_DIR = "models_nyc_taxi"

    MIN_TRIP_DURATION = 60
    MAX_TRIP_DURATION = 7200
    MIN_TRIP_DISTANCE = 0.1
    MAX_TRIP_DISTANCE = 50
    NYC_LAT_RANGE = (40.5, 40.9)
    NYC_LON_RANGE = (-74.3, -73.7)
    IQR_FACTOR = 1.5

    RF_N_ESTIMATORS = 100
    RF_MAX_DEPTH = 20
    RF_MIN_SAMPLES_SPLIT = 10

    GB_N_ESTIMATORS = 100
    GB_LEARNING_RATE = 0.1
    GB_MAX_DEPTH = 5

    RIDGE_ALPHA = 10.0
    LASSO_ALPHA = 0.1
    ELASTIC_ALPHA = 0.1
    ELASTIC_L1_RATIO = 0.5

    MODEL_NAME = "nyc_taxi_predictor"


config = Config()
os.makedirs(config.MODEL_DIR, exist_ok=True)

print(f"Random state: {config.RANDOM_STATE}")
print(f"Model name:   {config.MODEL_NAME}")

Random state: 42
Model name:   nyc_taxi_predictor


## 2. Load Data

Same DVC pull. But now we log the data hash as a parameter — so this run is anchored to a specific data version.

In [4]:
!dvc pull data/sample_nyc_taxi.csv.dvc

df = pd.read_csv("data/sample_nyc_taxi.csv")
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

# Read the DVC hash from the pointer file — this is the data version
with open("data/sample_nyc_taxi.csv.dvc") as f:
    dvc_pointer = f.read()

import re
dvc_hash_match = re.search(r"md5:\s*([a-f0-9]+)", dvc_pointer)
dvc_hash = dvc_hash_match.group(1) if dvc_hash_match else "unknown"

print(f"DVC data hash: {dvc_hash}")

Fetching
!
  0% Checking cache in '/home/silva/SILVA.AI/Projects/MLOps/DDODS/Module_2_Repr
Fetching                                                                       
Building workspace index                             |2.00 [00:00, 84.8entry/s]
Comparing indexes                                    |3.00 [00:00,  484entry/s]
Applying changes                                     |0.00 [00:00,     ?file/s]
Everything is up to date.
Loaded 100,000 rows × 19 columns
DVC data hash: 4a1073c07decf66c925d0a4e1b583165


## 3. Clean Data

In [5]:
def clean_nyc_taxi_data(df, config):
    df = df.copy()
    initial = len(df)

    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['trip_duration_minutes'] = (
        df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    ).dt.total_seconds() / 60

    df = df[
        (df['trip_duration_minutes'] >= config.MIN_TRIP_DURATION / 60) &
        (df['trip_duration_minutes'] <= config.MAX_TRIP_DURATION / 60)
    ]
    df = df[
        (df['trip_distance'] >= config.MIN_TRIP_DISTANCE) &
        (df['trip_distance'] <= config.MAX_TRIP_DISTANCE)
    ]
    df = df[
        df['pickup_latitude'].between(*config.NYC_LAT_RANGE) &
        df['pickup_longitude'].between(*config.NYC_LON_RANGE) &
        df['dropoff_latitude'].between(*config.NYC_LAT_RANGE) &
        df['dropoff_longitude'].between(*config.NYC_LON_RANGE)
    ]
    df = df[df['passenger_count'].between(1, 6)]
    df = df.dropna().reset_index(drop=True)

    return df, initial


df_clean, initial_rows = clean_nyc_taxi_data(df, config)
print(f"Cleaned: {initial_rows:,} → {len(df_clean):,} rows")

Cleaned: 100,000 → 97,050 rows


## 4. Define Features

In [6]:
PREDICTION_TIME_FEATURES = [
    'tpep_pickup_datetime',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
    'passenger_count', 'VendorID', 'RatecodeID',
    'trip_distance', 'payment_type',
]

LEAKAGE_FEATURES = [
    'fare_amount', 'tip_amount', 'total_amount', 'extra', 'mta_tax',
    'tolls_amount', 'improvement_surcharge', 'store_and_fwd_flag',
    'tpep_dropoff_datetime',
]

X = df_clean[PREDICTION_TIME_FEATURES]
y = df_clean['trip_duration_minutes'].values

print(f"X: {X.shape}, y: {y.shape}")

X: (97050, 10), y: (97050,)


## 5. Split Train / Val / Test

In [7]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=config.VAL_SIZE, random_state=config.RANDOM_STATE
)

split_info = {
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
}

print(f"Train: {len(X_train):,}")
print(f"Val:   {len(X_val):,}")
print(f"Test:  {len(X_test):,}")

Train: 62,112
Val:   15,528
Test:  19,410


## 6. Feature Engineering

In [8]:
class NYCYellowTaxiFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.feature_names_ = []

    def fit(self, X, y=None):
        return self

    @staticmethod
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 3958.8
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        return 2 * R * np.arcsin(np.sqrt(a))

    def transform(self, X):
        X = X.copy()
        X['tpep_pickup_datetime'] = pd.to_datetime(X['tpep_pickup_datetime'])
        X['haversine_distance'] = self.haversine_distance(
            X['pickup_latitude'], X['pickup_longitude'],
            X['dropoff_latitude'], X['dropoff_longitude'],
        )
        delta_lat = X['dropoff_latitude'] - X['pickup_latitude']
        delta_lon = X['dropoff_longitude'] - X['pickup_longitude']
        X['manhattan_distance'] = np.abs(delta_lat) * 69.0 + np.abs(delta_lon) * 53.0
        X['direction_sin'] = np.sin(np.arctan2(delta_lat, delta_lon))
        X['direction_cos'] = np.cos(np.arctan2(delta_lat, delta_lon))
        X['pickup_hour'] = X['tpep_pickup_datetime'].dt.hour
        X['pickup_dayofweek'] = X['tpep_pickup_datetime'].dt.dayofweek
        X['pickup_month'] = X['tpep_pickup_datetime'].dt.month
        X['hour_sin'] = np.sin(2 * np.pi * X['pickup_hour'] / 24)
        X['hour_cos'] = np.cos(2 * np.pi * X['pickup_hour'] / 24)
        X['dayofweek_sin'] = np.sin(2 * np.pi * X['pickup_dayofweek'] / 7)
        X['dayofweek_cos'] = np.cos(2 * np.pi * X['pickup_dayofweek'] / 7)
        X['is_rush_hour'] = (
            X['pickup_hour'].between(7, 9) | X['pickup_hour'].between(16, 18)
        ).astype(int)
        X['is_weekend'] = X['pickup_dayofweek'].isin([5, 6]).astype(int)
        X['is_night'] = ((X['pickup_hour'] >= 22) | (X['pickup_hour'] <= 5)).astype(int)
        landmarks = {
            'jfk': (40.6413, -73.7781),
            'lga': (40.7769, -73.8740),
            'times_square': (40.7580, -73.9855),
        }
        for name, (lat, lon) in landmarks.items():
            X[f'pickup_from_{name}'] = self.haversine_distance(
                X['pickup_latitude'], X['pickup_longitude'], lat, lon
            )
        X['is_jfk_trip'] = (X['pickup_from_jfk'] < 2).astype(int)
        X['is_lga_trip'] = (X['pickup_from_lga'] < 2).astype(int)
        X['efficiency_ratio'] = X['haversine_distance'] / (X['trip_distance'] + 1e-8)
        X['distance_per_passenger'] = X['trip_distance'] / (X['passenger_count'] + 1e-8)
        X['is_vendor_2'] = (X['VendorID'] == 2).astype(int)
        X['is_standard_rate'] = (X['RatecodeID'] == 1).astype(int)
        X['is_credit_card'] = (X['payment_type'] == 1).astype(int)
        X['distance_times_passengers'] = X['trip_distance'] * X['passenger_count']
        X['haversine_times_hour'] = X['haversine_distance'] * X['pickup_hour']
        X = X.drop(columns=[
            'tpep_pickup_datetime', 'VendorID', 'RatecodeID', 'payment_type'
        ], errors='ignore')
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.feature_names_ = numeric_cols
        return X[numeric_cols].values

    def get_feature_names(self):
        return self.feature_names_


class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = None
        self.upper_bounds_ = None

    def fit(self, X, y=None):
        self.lower_bounds_ = []
        self.upper_bounds_ = []
        for i in range(X.shape[1]):
            q1 = np.percentile(X[:, i], 25)
            q3 = np.percentile(X[:, i], 75)
            iqr = q3 - q1
            self.lower_bounds_.append(q1 - self.factor * iqr)
            self.upper_bounds_.append(q3 + self.factor * iqr)
        return self

    def transform(self, X):
        X = X.copy()
        for i in range(X.shape[1]):
            X[:, i] = np.clip(X[:, i], self.lower_bounds_[i], self.upper_bounds_[i])
        return X

## 7. Build Pipeline

In [9]:
preprocessor = Pipeline([
    ('feature_engineer', NYCYellowTaxiFeatureEngineer()),
    ('outlier_handler', OutlierHandler(factor=config.IQR_FACTOR)),
    ('scaler', RobustScaler()),
])

preprocessor.fit(X_train, y_train)

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

n_features = len(preprocessor.named_steps['feature_engineer'].get_feature_names())
print(f"Engineered features: {n_features}")

Engineered features: 32


## 8. Model Portfolio

In [10]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(random_state=config.RANDOM_STATE, alpha=config.RIDGE_ALPHA),
    'Lasso': Lasso(random_state=config.RANDOM_STATE, alpha=config.LASSO_ALPHA, max_iter=5000),
    'ElasticNet': ElasticNet(random_state=config.RANDOM_STATE, alpha=config.ELASTIC_ALPHA,
                            l1_ratio=config.ELASTIC_L1_RATIO, max_iter=5000),
    'Random Forest': RandomForestRegressor(
        random_state=config.RANDOM_STATE, n_jobs=config.N_JOBS,
        n_estimators=config.RF_N_ESTIMATORS, max_depth=config.RF_MAX_DEPTH,
        min_samples_split=config.RF_MIN_SAMPLES_SPLIT,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        random_state=config.RANDOM_STATE, n_estimators=config.GB_N_ESTIMATORS,
        learning_rate=config.GB_LEARNING_RATE, max_depth=config.GB_MAX_DEPTH,
    ),
}

print(f"{len(models)} models in the portfolio")

6 models in the portfolio


## 9. Train All Models — Each as an MLflow Run

This is where the tracking happens. Every model training becomes an MLflow run with:

- **Params** — the model's hyperparameters, data hash, split sizes, feature count
- **Metrics** — train/val R², RMSE, MAE, training time
- **Model artifact** — the fitted model, with an inferred signature

After this cell, the MLflow UI shows 6 runs. You can sort by `val_r2`, filter by `model_type`, compare them side by side.

In [11]:
results = {}
trained_models = {}
run_ids = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name) as run:
        # Train
        t0 = time.time()
        model.fit(X_train_processed, y_train)
        train_time = time.time() - t0

        # Evaluate
        y_train_pred = model.predict(X_train_processed)
        y_val_pred = model.predict(X_val_processed)

        metrics = {
            'train_r2': r2_score(y_train, y_train_pred),
            'val_r2': r2_score(y_val, y_val_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'val_rmse': np.sqrt(mean_squared_error(y_val, y_val_pred)),
            'train_mae': mean_absolute_error(y_train, y_train_pred),
            'val_mae': mean_absolute_error(y_val, y_val_pred),
            'training_time': train_time,
            'overfit_gap': r2_score(y_train, y_train_pred) - r2_score(y_val, y_val_pred),
        }

        # Log params
        params = {k: str(v) if callable(v) else v for k, v in model.get_params().items()}
        mlflow.log_params(params)

        # Log context
        mlflow.log_param('model_type', name)
        mlflow.log_param('data_version', dvc_hash)
        mlflow.log_param('train_samples', split_info['train_size'])
        mlflow.log_param('val_samples', split_info['val_size'])
        mlflow.log_param('test_samples', split_info['test_size'])
        mlflow.log_param('engineered_features', n_features)

        # Log metrics
        mlflow.log_metrics(metrics)

        # Tag the run
        mlflow.set_tag('model_family', name)
        mlflow.set_tag('data_source', 'dvc')

        # Log the model — trust the sklearn tree type
        signature = infer_signature(X_train_processed, y_train_pred)
        mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path='model',
        signature=signature,
        serialization_format='pickle',
    )

        results[name] = metrics
        trained_models[name] = model
        run_ids[name] = run.info.run_id

        print(f"{name:22}  val_r2={metrics['val_r2']:.4f}  "
              f"val_mae={metrics['val_mae']:6.2f} min  run_id={run.info.run_id[:8]}")

2026/10/04 22:23:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Linear Regression       val_r2=0.7399  val_mae=  2.94 min  run_id=f4dad003


2026/10/04 22:23:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Ridge                   val_r2=0.7400  val_mae=  2.94 min  run_id=75ecaa75


2026/10/04 22:23:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

ElasticNet              val_r2=0.7283  val_mae=  2.98 min  run_id=7ae49716


2026/10/04 22:23:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest           val_r2=0.8453  val_mae=  2.23 min  run_id=2283cc05


2026/10/04 22:25:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Gradient Boosting       val_r2=0.8450  val_mae=  2.24 min  run_id=3a00629b


## 10. Compare Models — Query MLflow, Don't Compute

The comparison table is now a query against the backend. Same data as the printed table in notebook 01, but pulled from storage instead of local memory.

In [12]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.val_r2 DESC'],
)

cols = [
    'tags.mlflow.runName',
    'metrics.val_r2', 'metrics.val_mae', 'metrics.val_rmse',
    'metrics.training_time', 'metrics.overfit_gap',
    'run_id',
]
view = runs_df[cols].rename(columns=lambda c: c.split('.')[-1])
print(view.round(4).to_string(index=False))

best_run = runs_df.iloc[0]
best_run_id = best_run['run_id']
best_model_name = best_run['tags.mlflow.runName']
best_model = trained_models[best_model_name]

print(f"\nBest: {best_model_name}  (run_id={best_run_id[:8]})")

          runName  val_r2  val_mae  val_rmse  training_time  overfit_gap                           run_id
    Random Forest  0.8453   2.2262    3.4596        22.6416       0.1017 2283cc05687647f8974ae7a8285f1269
Gradient Boosting  0.8450   2.2419    3.4639        90.4302       0.0157 3a00629b5f2e4a84b556a7093caa2b33
            Ridge  0.7400   2.9389    4.4860         0.0305      -0.0050 75ecaa7569d140aa80f98b48ff4f0e9a
Linear Regression  0.7399   2.9391    4.4861         0.1220      -0.0050 f4dad00360a84cfaa8422f21b5a5df48
       ElasticNet  0.7283   2.9754    4.5855         0.1133      -0.0059 7ae49716397c42209a518a0cd1ecae98
            Lasso  0.7273   2.9928    4.5942         0.1927      -0.0053 1b67cca457804d0b88e00c5359ff761d

Best: Random Forest  (run_id=2283cc05)


## 11. Tune the Best Model — Logged as a Nested Run

In [13]:
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [15, 20, None],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 2],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 150],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5],
        'subsample': [0.8, 1.0],
    },
}

best_run_id_after_tuning = best_run_id

if best_model_name in param_grids:
    with mlflow.start_run(run_name=f"{best_model_name}_tuned", nested=True) as tune_run:
        search = RandomizedSearchCV(
            models[best_model_name],
            param_grids[best_model_name],
            n_iter=5,                # reduced from 10
            cv=3,
            scoring='r2',
            n_jobs=2,                # reduced from -1
            random_state=config.RANDOM_STATE,
        )
        search.fit(X_train_processed, y_train)

        tuned_model = search.best_estimator_
        y_val_pred = tuned_model.predict(X_val_processed)

        mlflow.log_params(search.best_params_)
        mlflow.log_metric('cv_best_score', search.best_score_)
        mlflow.log_metric('val_r2', r2_score(y_val, y_val_pred))
        mlflow.log_metric('val_mae', mean_absolute_error(y_val, y_val_pred))

        signature = infer_signature(
            X_train_processed, tuned_model.predict(X_train_processed)
        )
        mlflow.sklearn.log_model(
            sk_model=tuned_model,
            artifact_path='model',
            signature=signature,
            serialization_format='pickle',
        )

        print(f"Best CV R²: {search.best_score_:.4f}")
        print(f"Best params: {search.best_params_}")

        best_model = tuned_model
        best_run_id_after_tuning = tune_run.info.run_id
else:
    print(f"{best_model_name} has no tuning grid. Skipping.")

2026/10/04 22:30:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best CV R²: 0.8318
Best params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 20}


## 12. Test Set Evaluation

Log the test metrics to the winning run. This closes the loop — the run now carries the full picture: training, validation, and final test performance.

In [14]:
y_test_pred = best_model.predict(X_test_processed)

test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"Test R²:   {test_r2:.4f}")
print(f"Test RMSE: {test_rmse:.2f} min")
print(f"Test MAE:  {test_mae:.2f} min")

# Log to the winning run
with mlflow.start_run(run_id=best_run_id_after_tuning):
    mlflow.log_metrics({
        'test_r2': test_r2,
        'test_rmse': test_rmse,
        'test_mae': test_mae,
    })
    mlflow.set_tag('final_model', 'true')

Test R²:   0.8422
Test RMSE: 3.52 min
Test MAE:  2.24 min


## 13. Register the Best Model

Registering assigns a **name** and a **version number** to a run's model. After this, the model isn't referenced by `run_id` anymore — it's referenced by name. `nyc_taxi_predictor` version 1.

In [15]:
model_uri = f"runs:/{best_run_id_after_tuning}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=config.MODEL_NAME,
)

print(f"Registered: {registered.name}  version {registered.version}")
print(f"Source run: {best_run_id_after_tuning[:8]}")

2026/10/04 22:30:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/04 22:30:46 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
Successfully registered model 'nyc_taxi_predictor'.
2026/10/04 22:30:47 WARNING mlflow.tracking._model_registry.fluent: Run with id 81c9961ef1e84fe1a55980baf98216ac has no artifacts at artifact path 'model', registering model based on models:/m-f67f1210ee5b48c6b0ad2549531e1513 instead


Registered: nyc_taxi_predictor  version 1
Source run: 81c9961e


Created version '1' of model 'nyc_taxi_predictor'.


## 14. Promote with Stages and Aliases

Two ways to mark a model as the active one:

- **Stages** — the classic approach. `Staging`, `Production`, `Archived`. One stage per version.
- **Aliases** — the modern approach. Named pointers. Multiple aliases per version.

We'll do both — first stage, then alias. Real projects use one or the other; you should know both.

In [16]:
version = registered.version

# ---- Classic: stages ----
client.transition_model_version_stage(
    name=config.MODEL_NAME,
    version=version,
    stage='Staging',
)
print(f"v{version} → Staging")

client.transition_model_version_stage(
    name=config.MODEL_NAME,
    version=version,
    stage='Production',
)
print(f"v{version} → Production")

# ---- Modern: aliases ----
client.set_registered_model_alias(
    name=config.MODEL_NAME,
    alias='champion',
    version=version,
)
print(f"v{version} → @champion")

v1 → Staging
v1 → Production
v1 → @champion


## 15. Load by Stage and Alias

The serving code doesn't need to know the version number. It loads by stage name or alias.

Same code today and next week, even when a new version replaces this one.

In [17]:
# Load by stage
model_by_stage = mlflow.sklearn.load_model(
    f"models:/{config.MODEL_NAME}/Production"
)

# Load by alias
model_by_alias = mlflow.sklearn.load_model(
    f"models:/{config.MODEL_NAME}@champion"
)

print(f"Loaded by stage: {type(model_by_stage).__name__}")
print(f"Loaded by alias: {type(model_by_alias).__name__}")

# Sanity prediction
sample = X_test_processed[:1]
prediction = model_by_alias.predict(sample)[0]
print(f"\nSample prediction: {prediction:.2f} minutes")

Loaded by stage: RandomForestRegressor
Loaded by alias: RandomForestRegressor

Sample prediction: 21.84 minutes


## 16. Save a Local Copy

The registry handles the lifecycle. But sometimes you still want the model on disk — for quick tests, or to ship to a serving system that expects files.

MLflow can download a registered model to a local directory, artifact contract intact.

In [18]:
from mlflow.artifacts import download_artifacts

local_dir = download_artifacts(
    artifact_uri=f"models:/{config.MODEL_NAME}/{version}",
    dst_path="model_dir",
)

print(f"Downloaded to: {local_dir}")

for f in sorted(os.listdir(local_dir)):
    print(f"  {f}")

Downloaded to: /home/silva/SILVA.AI/Projects/MLOps/DDODS/Module_2_Reproducibility/Lesson_2_Experiment_Tracking_MLflow/model_dir/
  MLmodel
  conda.yaml
  input_example.json
  model.pkl
  python_env.yaml
  registered_model_meta
  requirements.txt
  serving_input_example.json


## The Difference

Same pipeline. Same models. Same data. Same metrics.

But now:

- **Every run is in the MLflow UI.** Sortable by R². Filterable by model family. Comparable side by side.
- **Every model is versioned.** `nyc_taxi_predictor` version 1 exists as a named artifact, not a file path.
- **Every artifact is traceable.** Click any run in the UI → see its params, its metrics, its dataset hash, its model.
- **The serving code loads by stage or alias.** `models:/nyc_taxi_predictor/Production`. No file paths. No version numbers in code.
- **The next run is comparable to this one.** Run the notebook again tomorrow, and the UI shows both — sorted, filterable, comparable.

That's the difference between a notebook and a tracked experiment.